In [3]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
import pickle

make_train_test_split = True

if make_train_test_split:
    try:
        # 1. Load Data
        print("Loading 'final_merged_dataset.csv'...")
        final_df = pd.read_csv("./Health_data/final_merged_dataset.csv")
        final_df = final_df.dropna(subset=['InsptDate'])

        # 2. Define Target, Group Key, and Features
        target = 'Healthy'
        group_key = 'HiveID'
        
        # 2.1 Historical and Weather Features
        health_history = [
        'Is_First_Inspection', 'Days_Since_Last_Inspection', 'Hive_Age_Days',
        'Prev_Brood_Status', 'Prev_Bees_Status', 'Prev_Queen_Status',
        'Prev_Food_Status', 'Prev_Stressors_Status', 'Prev_Space_Status']

        weather_features = [
        'Avg_prcp', 'Avg_wind', 'Avg_tmax', 'Avg_tmin', 'Avg_tavg',
        'Avg_snow', 'Num_frost_days']

        health_weather_features = health_history + weather_features
        features_to_use = health_weather_features

        # 2.2 Checking if any health history features are 0.5 and setting them to 0
        for feature in health_history:
            if final_df[feature].dtype == float:
                final_df[feature] = final_df[feature].apply(lambda x: 0 if x == 0.5 else x)

        # 3. Create X, Y, and groups
        X = final_df[features_to_use]
        Y = final_df[target].astype(int)
        groups = final_df[group_key]

        # 4. Create Train/Test Split based on GroupSplit
        print("\nCreating GroupShuffleSplit...")
        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
        train_idx, test_idx = next(gss.split(X, Y, groups=groups))

        # 5. Create the final data splits
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        Y_train, Y_test = Y.iloc[train_idx], Y.iloc[test_idx]
        groups_train = groups.iloc[train_idx]
        groups_test = groups.iloc[test_idx] # Good to save this for analysis

        # 6. Save all files
        training_set = {
            "X_train": X_train,
            "Y_train": Y_train,
            "groups_train": groups_train}
        test_set = {
            "X_test": X_test,
            "Y_test": Y_test,
            "groups_test": groups_test}
        print("Saving files to disk...")
        with open('training_data.pkl', 'wb') as f:
            pickle.dump(training_set, f)
        with open('test_data.pkl', 'wb') as f:
            pickle.dump(test_set, f)

        print("\n--- Split Complete ---")
        print(f"X_train shape: {X_train.shape}")
        print(f"X_test shape: {X_test.shape}")
        print(f"Y_train shape: {Y_train.shape}")
        print(f"Y_test shape: {Y_test.shape}")
        print(f"groups_train shape: {groups_train.shape}")

    except Exception as e:
        print(f"An error occurred: {e}")
else:
    print("make_train_test_split is False. Loading existing files.")

Loading 'final_merged_dataset.csv'...

Creating GroupShuffleSplit...
Saving files to disk...

--- Split Complete ---
X_train shape: (1596, 16)
X_test shape: (491, 16)
Y_train shape: (1596,)
Y_test shape: (491,)
groups_train shape: (1596,)
